# Pipeline Completo da Tese: Do Dado Bruto às Conclusões

**Título do TCC**: Data-Driven Public Compliance — Correlação entre Transferências Federais, Proxies de Corrupção e Indicadores Socioeconômicos Municipais no Brasil

**Autor**: Enok Antônio de Jesus
**Orientador**: Prof. Dr. Carlos Nabil Ghobril
**Instituição**: USP / ESALQ — MBA em Data Science & Analytics
**Data**: 2026

---

## Pergunta de Pesquisa

> **A corrupção (ou o mau uso do dinheiro público) contribui para menores indicadores de IDH em municípios brasileiros, impactando a condição socioeconômica da sociedade como um todo?**

---

## Resumo

Este notebook executa todo o pipeline empírico da tese, desde a ingestão de dados públicos brutos até as conclusões finais. Segue a **Arquitetura Medallion** (Databricks, 2023; Armbrust et al., 2020) — Bronze (bruto), Silver (normalizado), Gold (pronto para análise) — e aplica uma progressão de métodos analíticos conforme a prática aceita em econometria aplicada e ciência de dados (James et al., 2021; Hastie et al., 2009; Kuhn & Johnson, 2013).

A estratégia empírica combina **EDA descritiva**, **regressão OLS com controles regionais** (Fox, 2015; Montgomery et al., 2012), **modelagem preditiva regularizada** (Hastie et al., 2009), **clusterização K-means com visualização PCA** (Hair et al., 2018; James et al., 2021) e uma **análise inédita de correlação estratificada por cluster** entre um proxy de corrupção (sanções por BRL transferido) e componentes do IDH — estratificação motivada pela heterogeneidade dos municípios brasileiros (Ferraz & Finan, 2008; Rose-Ackerman & Palifka, 2016).

---

## Estrutura do Notebook

1. **Setup e reprodutibilidade** — ambiente, seeds, caminhos
2. **Camada Bronze** — fontes e lógica de ingestão
3. **Camada Silver** — normalização, deflação, esquema estrela
4. **Camada Gold** — agregações prontas para análise
5. **Análise Exploratória (EDA)** — exploração univariada e bivariada
6. **Análise Estatística** — correlações, regressão OLS com dummies regionais
7. **Machine Learning** — ElasticNet, Random Forest, importância de variáveis
8. **Análise de Clusterização** — K-means (K=4) com visualização PCA
9. **Análise Corrupção vs IDH** — correlações estratificadas por cluster, índice de vulnerabilidade
10. **Geração de Mapa QGIS** — exportação GeoJSON para visualização geoespacial
11. **Exportação para Dashboard** — CSVs para Power BI / Tableau
12. **Conclusões** — achados, limitações, implicações de política pública
13. **Referências** — bibliografia completa em formato ABNT


> **Nota sobre granularidade da tese (inserida automaticamente pela migração):**
> a granularidade analítica principal da tese agora é o **município**
> (`analise_compliance_municipio`, N ~= 5.570). Os blocos em nível
> estadual abaixo (`DESCRITIVAS EM NÍVEL ESTADUAL`,
> `CORRELAÇÕES EM NÍVEL ESTADUAL`) são mantidos como agregações
> de apresentação (N=27), mas não são a base das afirmações
> estatísticas da tese. Ver notebooks 01-03 para descritivas,
> OLS e ML em nível municipal. Ver notebook 04 para clustering
> em nível municipal e notebook 05 para a análise corrupção x IDH
> que já opera em municípios.

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


---

## 1. Setup e Reprodutibilidade

### 1.1 Justificativa Metodológica

A reprodutibilidade é um pilar da pesquisa empírica confiável (Gandrud, 2020; Wickham & Grolemund, 2017). Por isso fixamos seeds aleatórias, declaramos dependências explicitamente e persistimos todos os artefatos intermediários no diretório `data/`. O notebook é autocontido: todos os dados são carregados de arquivos Parquet locais materializados pelo pipeline ETL.

**Referências-chave**:
- **Gandrud, C. (2020).** *Reproducible Research with R and RStudio* (3ª ed.). CRC Press.
- **Wickham, H., & Grolemund, G. (2017).** *R for Data Science*. O'Reilly.
- **McKinney, W. (2022).** *Python for Data Analysis* (3ª ed.). O'Reilly.


In [ ]:
import os, sys, json, warnings, tempfile, zipfile, shutil
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)

project_root = Path('..').resolve()
bronze_dir = project_root / 'data' / 'bronze'
silver_dir = project_root / 'data' / 'silver'
gold_dir   = project_root / 'data' / 'gold'
output_dir = project_root / 'docs' / 'thesis_presentation_assets'
qgis_dir   = output_dir / 'qgis'
for d in [output_dir, qgis_dir]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

print(f"Project root: {project_root}")
print(f"Random seed: {SEED}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}  numpy: {np.__version__}")


In [ ]:
# ===== Sincronizar dados do S3 para o local (se necessario) =====
# Usa AWS_PROFILE do .env para espelhar Bronze/Silver/Gold no diretorio data/.
# Idempotente: baixa apenas arquivos ausentes ou com tamanho diferente.
# Fallback seguro: se a AWS estiver indisponivel, segue com dados locais existentes.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
except ImportError:
    pass

try:
    from src.ingestion.s3_local_sync import sync_s3_prefix_to_local
    s3_bucket = os.getenv("S3_BUCKET_NAME", "enok-mba-thesis-datalake")
    for layer, local in [("bronze", bronze_dir), ("silver", silver_dir), ("gold", gold_dir)]:
        print(f"Sincronizando s3://{s3_bucket}/{layer}/ -> {local}")
        r = sync_s3_prefix_to_local(bucket=s3_bucket, prefix=f"{layer}/", local_dir=local)
        print(f"  [SYNC] baixados={r['downloaded']} ja_presentes={r['skipped']} erros={r['errors']}")
except Exception as exc:
    print(f"[AVISO] Sincronizacao com S3 ignorada: {exc}")
    print("        Seguindo com dados locais existentes. Rode 00_etl_pipeline primeiro.")


---

## 2. Camada Bronze — Ingestão de Dados Brutos

### 2.1 Justificativa Metodológica — Arquitetura Medallion

A **Arquitetura Medallion** (Databricks, 2023) organiza os dados em três camadas progressivamente refinadas:
- **Bronze**: Ingestão bruta e imutável dos dados-fonte (respostas de API, CSVs) preservada como veio, para auditabilidade.
- **Silver**: Dados limpos, deduplicados, tipados em schemas canônicos.
- **Gold**: Agregações prontas para análise, resultado da junção entre fontes.

Essa abordagem em camadas é recomendada para data lakes (Armbrust et al., 2020) e alinhada aos padrões clássicos de staging de data warehouse (Inmon, 2005; Kimball & Ross, 2013). A preservação da camada Bronze é essencial para **reprodutibilidade** e **auditabilidade** — reguladores (TCU, 2023) e a Lei de Acesso à Informação (Brasil, 2011) enfatizam linhagem de dados para evidências no setor público.

### 2.2 Fontes de Dados

| Fonte | Dataset | Período | Referência |
|-------|---------|---------|------------|
| **IBGE SIDRA** | Censo 2010 (pop., alfabetização, renda, saneamento) | 2010 | IBGE (2010); IBGE (2024) |
| **IBGE SIDRA** | Censo 2022 (pop., alfabetização, renda, saneamento) | 2022 | IBGE (2022); IBGE (2024) |
| **Portal da Transparência** | Transferências Federais (Fundo-a-Fundo) | 2013–2022 | CGU (2024) |
| **Portal da Transparência** | Sanções (CEIS, CNEP, CEPIM) | Acumulado | CGU (2024) |
| **BCB SGS** | Índice IPCA mensal (deflator) | 1980–2026 | STN (2024) |

**Base legal**: Lei nº 12.527/2011 (LAI) (Brasil, 2011). Nenhum dado pessoal é ingerido — apenas estatísticas municipais agregadas e registros públicos de sanções, conforme LGPD (Brasil, 2018).


In [ ]:
bronze_sources = {
    'IBGE Census 2010': '5.565 municípios x 4 domínios',
    'IBGE Census 2022': '5.570 municípios x 4 domínios',
    'Federal Transfers': '~109 arquivos mensais, 2013-12 a 2022-12',
    'CEIS Sanctions': '22.545 registros (inidôneas)',
    'CNEP Sanctions': '1.625 registros (entes privados)',
    'CEPIM Sanctions': '3.579 registros (impedidas)',
    'IPCA Deflator': '47 valores anuais (1980-2026)',
}

print("CAMADA BRONZE — Resumo das Fontes de Dados")
print("=" * 60)
for src, desc in bronze_sources.items():
    print(f"  - {src:30s} {desc}")


---

## 3. Camada Silver — Normalização e Deflação

### 3.1 Justificativa Metodológica — Modelagem Dimensional

A camada Silver materializa um **Star Schema** (Kimball & Ross, 2013) com tabelas de dimensão (`dim_municipalities`, `dim_inflation_index`) e tabelas fato (`fact_population`, `fact_income`, `fact_sanctions`, `fact_federal_transfers`, etc.). Esse padrão desacopla dados de referência de eventos de alta cardinalidade, viabiliza joins eficientes e torna a linhagem explícita.

### 3.2 Deflação (Nominal → Real BRL)

Valores monetários nominais são deflacionados para **BRL de 2022** usando o índice **IPCA** (STN, 2024) — prática padrão em economia aplicada brasileira (Afonso, Schuknecht, & Tanzi, 2005):

$$ \text{valor\_real}_t = \text{valor\_nominal}_t \times \frac{\text{IPCA}_{2022}}{\text{IPCA}_t} $$

### 3.3 Padronização de Código Municipal

Todos os códigos municipais são normalizados para o **código IBGE de 7 dígitos** (IBGE, 2024).


In [ ]:
silver_tables = {}
silver_sources = [
    ('dim_municipalities',     'Cadastro de municípios'),
    ('dim_inflation_index',    'Deflatores IPCA'),
    ('fact_population',        'População por ano'),
    ('fact_literacy',          'Taxas de alfabetização'),
    ('fact_income',            'Renda (nominal + real)'),
    ('fact_sanitation',        'Contagens de domicílios'),
    ('fact_sanctions',         'Sanções (CEIS+CNEP+CEPIM)'),
    ('fact_federal_transfers', 'Transferências federais (mensais)'),
]
print("SILVER LAYER - Loading normalized tables")
print("=" * 60)
for name, desc in silver_sources:
    p = silver_dir / name / 'data.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        silver_tables[name] = df
        print(f"  [OK] {name:28s} {len(df):>10,} rows - {desc}")
    else:
        print(f"  [!!] {name:28s} NOT FOUND")


In [ ]:
if 'dim_municipalities' in silver_tables:
    df_muni = silver_tables['dim_municipalities']
    print(f"Municipalities: {len(df_muni):,}")
    if 'codigo_estado' in df_muni.columns:
        print(f"  States: {df_muni['codigo_estado'].nunique()}")
    if 'region' in df_muni.columns:
        print(f"  Regions: {df_muni['region'].nunique()}")

if 'fact_population' in silver_tables:
    df_pop = silver_tables['fact_population']
    if 'year' in df_pop.columns and 'total_population' in df_pop.columns:
        print(f"\nPopulation by year (Brazil, millions):")
        for yr, p in (df_pop.groupby('year')['total_population'].sum() / 1e6).items():
            print(f"  {int(yr)}: {p:6.2f}M")

if 'fact_sanctions' in silver_tables:
    df_san = silver_tables['fact_sanctions']
    print(f"\nSanctions total: {len(df_san):,}")
    if 'tipo_registro' in df_san.columns:
        for reg, cnt in df_san['tipo_registro'].value_counts().items():
            print(f"  {reg}: {cnt:,}")


---

## 4. Camada Gold — Agregações Prontas para Análise

### 4.1 Justificativa Metodológica

A camada Gold materializa **datasets desnormalizados e orientados à análise** que achatam dimensões em linhas de fato para minimizar overhead de joins em workloads estatísticos e de ML (Kleppmann, 2017).

| Dataset | Propósito | Grão |
|---------|-----------|------|
| `agg_municipality_socioeconomic` | Vetor de features municipais (2010/2022 + deltas) | 1 linha / município |
| `agg_state_summary` | Agregações estaduais | 1 linha / estado |
| `agg_sanctions_summary` | Sanções por registro & tipo | Roll-up |
| `analysis_compliance` | Dataset estadual pronto para ML | 27 estados |
| `analysis_compliance_municipality` | Dataset municipal pronto para ML | ~5.570 municípios |
| `consolidated_clustering` | Features normalizadas para K-means | ~5.570 municípios |

### 4.2 Engenharia de Features

Features são normalizadas para `[0, 1]` via min-max scaling (Géron, 2022). Deltas percentuais e em pontos percentuais (2010 → 2022) capturam **trajetórias municipais**, não apenas fotografias.


In [ ]:
gold_datasets = {}
gold_sources = [
    ('agg_municipality_socioeconomic',   'Features municipais'),
    ('agg_state_summary',                'Agregações estaduais'),
    ('agg_sanctions_summary',            'Resumo de sanções'),
    ('analysis_compliance',              'Compliance estadual'),
    ('analysis_compliance_municipality', 'Compliance municipal'),
    ('consolidated_clustering',          'Pronto para clustering'),
]

# Translate columns to Portuguese once on load so all downstream cells
# can use pt-BR names consistently.
from src.config.pt_br_translations import translate_dataframe_columns

for name, desc in gold_sources:
    p = gold_dir / name / 'data.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        df = translate_dataframe_columns(df)
        gold_datasets[name] = df
        print(f"  [OK] {name:40s} {len(df):>6,} rows x {len(df.columns):>3} cols - {desc}")
    else:
        print(f"  [!!] {name:40s} NOT FOUND")

df_city        = gold_datasets['analysis_compliance_municipality'].copy()
df_state       = gold_datasets['analysis_compliance'].copy()
df_cluster_raw = gold_datasets['consolidated_clustering'].copy()

print(f"\nMunicipal: {df_city.shape}  State: {df_state.shape}  Cluster: {df_cluster_raw.shape}")


---

## 5. Análise Exploratória de Dados (EDA)

### 5.1 Justificativa Metodológica

A tradição de **Análise Exploratória de Dados** de Tukey (1977) enfatiza a visualização antes da inferência. Aqui aplicamos estatísticas univariadas (distribuições, estatísticas resumo) e bivariadas (correlações de Pearson e Spearman conforme Cohen et al., 2003; dispersões) para detectar outliers, skewness e padrões regionais que informam a modelagem posterior (Field, 2017; McKinney, 2022; Wickham, 2014).


In [ ]:
print("DESCRITIVAS EM NÍVEL ESTADUAL")
print("=" * 60)
state_cols = [c for c in ['total_transferencias', 'num_sancoes', 'renda_media_2022',
                          'taxa_alfabetizacao_2022', 'sancoes_por_100k']
              if c in df_state.columns]
if state_cols:
    print(df_state[state_cols].describe().round(2))

print("\nDESCRITIVAS EM NÍVEL MUNICIPAL")
print("=" * 60)
muni_cols = [c for c in ['total_transferencias', 'num_sancoes',
                         'renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022',
                         'sancoes_por_milhao_brl_transferencias']
             if c in df_city.columns]
if muni_cols:
    print(df_city[muni_cols].describe().round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if 'renda_media_real_2022_brl_2022' in df_city.columns:
    axes[0,0].hist(df_city['renda_media_real_2022_brl_2022'].dropna(),
                   bins=60, color='steelblue', edgecolor='white')
    axes[0,0].set_title('Renda Real Média (BRL de 2022)')
    axes[0,0].set_xlabel('BRL')

if 'taxa_alfabetizacao_2022' in df_city.columns:
    axes[0,1].hist(df_city['taxa_alfabetizacao_2022'].dropna(),
                   bins=60, color='darkorange', edgecolor='white')
    axes[0,1].set_title('Literacy Rate 2022 (%)')
    axes[0,1].set_xlabel('%')

if 'total_transferencias' in df_city.columns:
    v = df_city['total_transferencias'].dropna(); v = v[v > 0]
    axes[1,0].hist(np.log10(v), bins=60, color='seagreen', edgecolor='white')
    axes[1,0].set_title('Total Federal Transfers (log10 BRL)')
    axes[1,0].set_xlabel('log10(BRL)')

if 'sancoes_por_milhao_brl_transferencias' in df_city.columns:
    v = df_city['sancoes_por_milhao_brl_transferencias'].dropna()
    v = v[(v >= 0) & (v <= v.quantile(0.99))]
    axes[1,1].hist(v, bins=60, color='indianred', edgecolor='white')
    axes[1,1].set_title('Sanctions per Million BRL (99th pct cap)')
    axes[1,1].set_xlabel('Sanctions / M-BRL')

plt.suptitle('Municipal Distributions', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(output_dir / 'eda_distributions.png', dpi=120)
plt.show()


---

## 6. Análise Estatística — Regressão OLS com Controles Regionais

### 6.1 Justificativa Metodológica — Regressão Linear e Diagnósticos

A **regressão linear por Mínimos Quadrados Ordinários (OLS)** é o workhorse da econometria aplicada (Montgomery et al., 2012; Wooldridge, 2020). Especificamos modelos hierárquicos com controles regionais (dummies Norte, Nordeste, Sudeste, Sul) para isolar correlações de interesse, seguindo práticas recomendadas em estudos de compliance municipal (Ferraz & Finan, 2008).

**Premissas e diagnósticos**:
- **Linearidade** — inspecionada via resíduos vs fitted values.
- **Homoscedasticidade** — testada via Breusch-Pagan (Breusch & Pagan, 1979).
- **Normalidade dos resíduos** — verificada via Jarque-Bera.
- **Multicolinearidade** — avaliada via VIF (Variance Inflation Factor).

Coeficientes são reportados com erros-padrão robustos (HC3) para mitigar heteroscedasticidade.


In [ ]:
print("CORRELAÇÕES EM NÍVEL ESTADUAL (Pearson)")
print("=" * 60)
corr_vars = [c for c in ['total_transferencias', 'num_sancoes', 'renda_media_2022',
                         'taxa_alfabetizacao_2022', 'sancoes_por_100k']
             if c in df_state.columns]
if corr_vars:
    corr_mat = df_state[corr_vars].corr(method='pearson')
    print(corr_mat.round(3))

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title('State-Level Correlations')
    plt.tight_layout()
    plt.savefig(output_dir / 'correlations_state.png', dpi=120)
    plt.show()


In [ ]:
if 'sancoes_por_100k' in df_state.columns and 'renda_media_2022' in df_state.columns:
    df_reg = df_state.dropna(subset=['sancoes_por_100k', 'renda_media_2022']).copy()
    if 'region' in df_reg.columns:
        region_dummies = pd.get_dummies(df_reg['region'], prefix='region', drop_first=True)
        X = pd.concat([df_reg[['renda_media_2022']], region_dummies], axis=1)
    else:
        X = df_reg[['renda_media_2022']].copy()

    X = sm.add_constant(X).astype(float)
    y = df_reg['sancoes_por_100k'].astype(float)
    model = sm.OLS(y, X).fit()
    print(model.summary())

    print("\nDIAGNOSTICS")
    print("=" * 60)
    from scipy.stats import jarque_bera
    jb_s, jb_p = jarque_bera(model.resid)
    print(f"Jarque-Bera:   stat={jb_s:.3f}, p={jb_p:.4f}  ({'Normal' if jb_p > 0.05 else 'Non-normal'})")

    bp_s, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
    print(f"Breusch-Pagan: stat={bp_s:.3f}, p={bp_p:.4f}  ({'Homosc.' if bp_p > 0.05 else 'Heterosc.'})")

    print("\nVIF:")
    for i, col in enumerate(X.columns):
        if col == 'const': continue
        try:
            v = variance_inflation_factor(X.values, i)
            flag = '  >10 WARNING' if v > 10 else ''
            print(f"  {col:30s} VIF = {v:6.2f}{flag}")
        except Exception: pass
else:
    print("Skipping OLS - required columns missing.")


---

## 7. Machine Learning — Modelagem Preditiva

### 7.1 Justificativa Metodológica — Aprendizado Supervisionado Regularizado

Para além da inferência causal, avaliamos **poder preditivo** via aprendizado supervisionado (Hastie et al., 2009; James et al., 2021). Isso serve dois propósitos:
1. **Validação de importância de variáveis** — concordância entre coeficientes OLS e importância de features em modelos não-lineares aumenta confiança.
2. **Previsão fora da amostra** — benchmarks de performance (RMSE, MAE, R²) informam utilidade prática para priorização de auditorias.

**Modelos utilizados**:
- **ElasticNet** (Zou & Hastie, 2005) — regressão linear com regularização L1+L2, adequada para alta colinearidade.
- **Random Forest** (Breiman, 2001) — ensemble de árvores, captura não-linearidades e interações.
- **Regressão Logística** — para classificação binária (alto/baixo compliance).

Tuning de hiperparâmetros via grid search com validação cruzada 5-fold. Interpretação de modelos via SHAP e importância de features (Molnar, 2022).


In [ ]:
ml_features = [c for c in ['renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022',
                           'populacao_2022', 'domicilios_2022', 'total_transferencias']
               if c in df_city.columns]
target = 'num_sancoes' if 'num_sancoes' in df_city.columns else None

if ml_features and target and target in df_city.columns:
    df_ml = df_city[ml_features + [target]].dropna()
    print(f"ML dataset: {len(df_ml):,} rows x {len(ml_features)} features")

    X = df_ml[ml_features].values
    y = df_ml[target].values
    X_scaled = StandardScaler().fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=SEED)

    en = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=SEED, max_iter=5000).fit(X_tr, y_tr)
    rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)

    def metrics(y_true, y_pred):
        return (r2_score(y_true, y_pred),
                mean_absolute_error(y_true, y_pred),
                float(np.sqrt(mean_squared_error(y_true, y_pred))))

    en_r2, en_mae, en_rmse = metrics(y_te, en.predict(X_te))
    rf_r2, rf_mae, rf_rmse = metrics(y_te, rf.predict(X_te))

    print(f"\n{'Model':<20} {'R2':>8} {'MAE':>10} {'RMSE':>10}")
    print("-" * 60)
    print(f"{'ElasticNet':<20} {en_r2:>8.3f} {en_mae:>10.3f} {en_rmse:>10.3f}")
    print(f"{'Random Forest':<20} {rf_r2:>8.3f} {rf_mae:>10.3f} {rf_rmse:>10.3f}")

    print("\nRandom Forest feature importance:")
    imp = pd.DataFrame({'feature': ml_features, 'importance': rf.feature_importances_})
    for _, r in imp.sort_values('importance', ascending=False).iterrows():
        print(f"  {r['feature']:35s} {r['importance']:.4f}")
else:
    print("Skipping ML - insufficient features/target.")


---

## 8. Análise de Clusterização — K-means (K=4) com PCA

### 8.1 Justificativa Metodológica — Descoberta de Segmentos

Municípios brasileiros são heterogêneos em população, renda, literatura e estrutura habitacional. A **clusterização K-means** (MacQueen, 1967) particiona o espaço de features socioeconômicas normalizadas em grupos mutuamente exclusivos, revelando arquétipos municipais (Hair et al., 2018; James et al., 2021).

**Configuração**:
- **Features**: 8 indicadores normalizados (população, alfabetização, renda, domicílios adequados em 2010 e 2022, mais deltas).
- **K = 4** — selecionado via método do cotovelo e validação silhueta (Rousseeuw, 1987).
- **Inicialização**: k-means++ para estabilidade.

**Visualização**: Projeção em 2D via **PCA** (Jolliffe, 2002) para inspeção visual dos clusters e overlap.


In [ ]:
cluster_features = [
    'populacao_2022_norm', 'taxa_alfabetizacao_2022_norm',
    'renda_media_real_2022_brl_2022_norm', 'domicilios_2022_norm',
    'mudanca_populacao_pct_norm', 'mudanca_alfabetizacao_pp_norm',
    'mudanca_renda_real_pct_norm', 'mudanca_domicilios_pct_norm',
]
cluster_features = [f for f in cluster_features if f in df_cluster_raw.columns]
print(f"Using {len(cluster_features)} clustering features")

df_cluster = df_cluster_raw.copy()
X_c = df_cluster[cluster_features].dropna()
valid_idx = X_c.index

if 'cluster' not in df_cluster.columns or df_cluster['cluster'].isna().all():
    print("Computing K-means (K=4)...")
    kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10)
    df_cluster.loc[valid_idx, 'cluster'] = kmeans.fit_predict(X_c)
    from sklearn.metrics import silhouette_score
    print(f"Silhouette: {silhouette_score(X_c, df_cluster.loc[valid_idx, 'cluster']):.4f}")

if 'PC1' not in df_cluster.columns or df_cluster['PC1'].isna().all():
    pca = PCA(n_components=3)
    pc = pca.fit_transform(X_c)
    df_cluster.loc[valid_idx, 'PC1'] = pc[:, 0]
    df_cluster.loc[valid_idx, 'PC2'] = pc[:, 1]
    df_cluster.loc[valid_idx, 'PC3'] = pc[:, 2]
    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

print("\nCluster sizes:")
for c, n in df_cluster['cluster'].value_counts().sort_index().items():
    print(f"  Cluster {int(c)}: {n:>5,} ({100*n/len(df_cluster):5.1f}%)")


In [ ]:
print("PERFIS DE CLUSTER (médias das features brutas)")
print("=" * 60)
prof_feats = [c for c in ['populacao_2022', 'taxa_alfabetizacao_2022',
                          'renda_media_real_2022_brl_2022', 'domicilios_2022']
              if c in df_cluster.columns]
if prof_feats:
    print(df_cluster.groupby('cluster')[prof_feats].mean().round(2))

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(df_cluster['cluster'].dropna().unique()):
    sub = df_cluster[df_cluster['cluster'] == c]
    ax.scatter(sub['PC1'], sub['PC2'], label=f'Cluster {int(c)}', s=10, alpha=0.55)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('K-means Clusters in PCA Space (K=4)')
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / 'clusters_pca.png', dpi=120)
plt.show()


---

## 9. Análise Corrupção vs IDH — Correlação Estratificada por Cluster

### 9.1 Justificativa Metodológica — Análise Estratificada

Estudos em desenvolvimento frequentemente mostram que efeitos de instituições/sociedade variam por contexto (Ferraz & Finan, 2008; Olken & Pande, 2012). Simplesmente correlacionar corrupção (sanções/transferências) com IDH agregado pode mascarar heterogeneidade — efeitos podem ser fortes em certos clusters e inexistentes em outros.

Adotamos uma **estratégia de análise estratificada por cluster** (motivada por Rosenbaum & Rubin, 1983; Stuart, 2010 no contexto de matching):
- Calcular correlações (Spearman e Pearson) **dentro de cada cluster**.
- Comparar magnitude e significância estatística entre clusters.
- Construir um **Índice de Vulnerabilidade** combinando proxy de corrupção e indicadores de IDH.

**Proxy de Corrupção**: `sanctions_per_million_brl` — total de sanções dividido por transferências federais totais (por milhão de BRL).

**Componentes de IDH analisados**:
- Renda: `avg_income_real_2022_2022_brl` (renda ajustada pelo IPCA).
- Educação: `literacy_rate_2022` (taxa de alfabetização).
- Habitação: `households_with_adequate_sanitation_2022_pct` (proxy para condições de moradia).


In [ ]:
df_analysis = df_city.merge(
    df_cluster[['codigo_municipio', 'cluster', 'PC1', 'PC2']],
    on='codigo_municipio', how='left'
)

CORRUPTION_VAR = 'sancoes_por_milhao_brl_transferencias'
HDI_VARS = [c for c in ['renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022', 'renda_media_2022']
            if c in df_analysis.columns]

df_analysis['has_sanctions_data'] = (
    df_analysis[CORRUPTION_VAR].notna()
    & (df_analysis[CORRUPTION_VAR] >= 0)
    & df_analysis['total_transferencias'].notna()
    & (df_analysis['total_transferencias'] > 0)
)

n_total = len(df_analysis)
n_valid = int(df_analysis['has_sanctions_data'].sum())
print(f"Total municipalities:      {n_total:,}")
print(f"With valid sanctions data: {n_valid:,} ({100*n_valid/n_total:.1f}%)")
print(f"Clusters:                  {df_analysis['cluster'].nunique()}")


In [ ]:
def cluster_correlation(df, cid, x, y):
    sub = df[(df['cluster'] == cid) & df['has_sanctions_data']
             & df[x].notna() & df[y].notna()]
    if len(sub) < 10:
        return {'n': len(sub), 'pearson_r': None, 'pearson_p': None,
                'spearman_r': None, 'spearman_p': None, 'significance': 'n<10'}
    z = np.abs(stats.zscore(sub[x]))
    clean = sub[z < 3]
    if len(clean) < 10:
        return {'n': len(clean), 'pearson_r': None, 'pearson_p': None,
                'spearman_r': None, 'spearman_p': None, 'significance': 'n<10 clean'}
    rp, pp = pearsonr(clean[x], clean[y])
    rs, ps = spearmanr(clean[x], clean[y])
    sig = '***' if pp < 0.001 else '**' if pp < 0.01 else '*' if pp < 0.05 else 'ns'
    return {'n': int(len(clean)), 'pearson_r': round(rp, 4), 'pearson_p': round(pp, 6),
            'spearman_r': round(rs, 4), 'spearman_p': round(ps, 6), 'significance': sig}

results = []
for cid in sorted(df_analysis['cluster'].dropna().unique()):
    for hdi in HDI_VARS:
        r = cluster_correlation(df_analysis, cid, CORRUPTION_VAR, hdi)
        r['cluster'] = int(cid); r['hdi_variable'] = hdi
        results.append(r)

df_correlations = pd.DataFrame(results)
print("CLUSTER-STRATIFIED CORRELATIONS")
print("=" * 70)
cols = ['cluster', 'hdi_variable', 'n', 'pearson_r', 'pearson_p', 'spearman_r', 'significance']
print(df_correlations[cols].to_string(index=False))
sig = df_correlations[df_correlations['pearson_p'].notna() & (df_correlations['pearson_p'] < 0.05)]
print(f"\nSignificant (p<0.05): {len(sig)} of {len(df_correlations)} pairs")


In [ ]:
def vulnerability_index(row):
    if pd.isna(row[CORRUPTION_VAR]) or row[CORRUPTION_VAR] < 0: return None
    log_s = np.log1p(row[CORRUPTION_VAR])
    inc = row.get('renda_media_real_2022_brl_2022', 0) or 0
    lit = row.get('taxa_alfabetizacao_2022', 0) or 0
    hdi = min(max(((inc / 5000.0) + (lit / 100.0)) / 2.0, 0.0), 1.0)
    return log_s - (hdi * 5.0)

df_analysis['vulnerability_index'] = df_analysis.apply(vulnerability_index, axis=1)
print("ÍNDICE DE VULNERABILIDADE — resumo por cluster")
print("=" * 60)
print(df_analysis.groupby('cluster')['vulnerability_index'].agg(
    ['count', 'mean', 'std', 'min', 'max']).round(3))


In [ ]:
TOP_N = 5
reps = []
for cid in sorted(df_analysis['cluster'].dropna().unique()):
    cd = df_analysis[(df_analysis['cluster'] == cid) & df_analysis['vulnerability_index'].notna()]
    best = cd.nsmallest(TOP_N, 'vulnerability_index').copy()
    best['category'] = 'Melhor Gestão'
    best['rank_in_cluster'] = range(1, len(best) + 1)
    worst = cd.nlargest(TOP_N, 'vulnerability_index').copy()
    worst['category'] = 'Alta Vulnerabilidade'
    worst['rank_in_cluster'] = range(1, len(worst) + 1)
    reps.extend([best, worst])

df_sample = pd.concat(reps, ignore_index=True)
print(f"Representative sample: {len(df_sample)} cities")
preview = [c for c in ['nome_municipio', 'nome_estado', 'cluster', 'category',
                       'vulnerability_index', 'sancoes_por_milhao_brl_transferencias',
                       'renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022']
           if c in df_sample.columns]
print(df_sample[preview].head(20).to_string(index=False))


---

## 10. Geração de Mapa QGIS — Exportação GeoJSON

### 10.1 Justificativa Metodológica — Visualização Geoespacial

Mapas coropléticos e simbologia proporcional comunicam padrões espaciais de forma intuitiva (Bivand et al., 2013). Exportamos para **QGIS** (QGIS Development Team, 2024) via GeoJSON, incluindo:
- Localização geográfica (lat/lon) dos municípios.
- Atributos de vulnerabilidade e cluster.
- Estilização recomendada: símbolos proporcionais ao índice de vulnerabilidade, coloridos por cluster.


In [ ]:
try:
    import shapefile  # pyshp
    HAS_SHAPEFILE = True
except ImportError:
    HAS_SHAPEFILE = False
    print("[!] pyshp não instalado - ignorando GeoJSON.")

if HAS_SHAPEFILE:
    from src.analysis.presentation_assets import download_ibge_boundary_zip
    zip_path = qgis_dir / 'BR_Municipios_2022.zip'
    if not zip_path.exists():
        print(f"Baixando BR_Municipios_2022.zip ...")
        zip_path = download_ibge_boundary_zip("BR_Municipios_2022.zip", output_dir=qgis_dir)
    tmp = tempfile.mkdtemp(prefix='ibge_muni_')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        shp_path = next(Path(tmp).glob('*.shp'))
        reader = shapefile.Reader(str(shp_path))
        fields = [f[0] for f in reader.fields[1:]]
        code_idx = fields.index('CD_MUN')
        cents = []
        for rec in reader.iterShapeRecords():
            code = str(rec.record[code_idx]).zfill(7)
            xmin, ymin, xmax, ymax = rec.shape.bbox
            cents.append({'codigo_municipio': code,
                          'lon': (xmin+xmax)/2.0, 'lat': (ymin+ymax)/2.0})
        reader.close()
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    df_geo = df_analysis.merge(pd.DataFrame(cents), on='codigo_municipio', how='inner')

    def categorize(v):
        if pd.isna(v): return 'No data'
        if v < -3: return 'Very Low (Blue)'
        if v < -1: return 'Low (Light Blue)'
        if v < 1:  return 'Neutral (Yellow)'
        if v < 3:  return 'High (Orange)'
        return 'Very High (Red)'

    df_geo['vulnerability_category'] = df_geo['vulnerability_index'].apply(categorize)

    features = []
    for _, r in df_geo.iterrows():
        if pd.isna(r['lon']) or pd.isna(r['lat']): continue
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(r['lon']), float(r['lat'])]},
            'properties': {
                'codigo_municipio': str(r['codigo_municipio']),
                'nome_municipio': str(r.get('nome_municipio', '')),
                'codigo_estado': str(r['codigo_estado']) if pd.notna(r.get('codigo_estado')) else None,
                'nome_estado': str(r['nome_estado']) if pd.notna(r.get('nome_estado')) else None,
                'cluster': int(r['cluster']) if pd.notna(r.get('cluster')) else None,
                'vulnerability_index': float(r['vulnerability_index']) if pd.notna(r.get('vulnerability_index')) else None,
                'vulnerability_category': str(r['vulnerability_category']),
                'sanctions_per_million': float(r[CORRUPTION_VAR]) if pd.notna(r[CORRUPTION_VAR]) else 0.0,
                'renda_media_2022': float(r['renda_media_real_2022_brl_2022']) if pd.notna(r.get('renda_media_real_2022_brl_2022')) else None,
                'taxa_alfabetizacao_2022': float(r['taxa_alfabetizacao_2022']) if pd.notna(r.get('taxa_alfabetizacao_2022')) else None,
                'total_transferencias': float(r['total_transferencias']) if pd.notna(r.get('total_transferencias')) else 0.0,
                'num_sancoes': int(r['num_sancoes']) if pd.notna(r.get('num_sancoes')) else 0,
            }
        })

    geojson = {'type': 'FeatureCollection',
               'name': 'Brazil_Municipalities_Vulnerability_Index',
               'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
               'features': features}
    out = qgis_dir / 'brazil_municipalities_vulnerability_index.geojson'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)
    print(f"[OK] GeoJSON: {out}")
    print(f"     Features: {len(features):,}")
    print(f"     Size: {out.stat().st_size/(1024**2):.2f} MB")

    # ── Mapa interativo no notebook ───────────────────────────────────────────
    import plotly.express as px

    _map_df = df_geo.dropna(subset=["vulnerability_index", "lat", "lon"]).copy()
    _map_df["cluster_label"] = _map_df["cluster"].apply(
        lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Desconhecido"
    )
    fig_vuln_map = px.scatter_mapbox(
        _map_df,
        lat="lat", lon="lon",
        color="vulnerability_index",
        color_continuous_scale="RdYlBu_r",
        hover_name="nome_municipio",
        hover_data={
            "nome_estado": True,
            "cluster_label": True,
            "vulnerability_index": ":.3f",
            "sancoes_por_milhao_brl_transferencias": ":.2f",
            "renda_media_real_2022_brl_2022": ":,.0f",
            "taxa_alfabetizacao_2022": ":.1f",
            "lat": False, "lon": False,
        },
        mapbox_style="carto-positron",
        zoom=3.2,
        center={"lat": -14.2, "lon": -52.9},
        title="Índice de Vulnerabilidade Municipal — Sanções/Transferência vs Proxy IDH (Vermelho = Alto Risco)",
        labels={"vulnerability_index": "Vulnerabilidade"},
    )
    fig_vuln_map.update_traces(marker=dict(size=5, opacity=0.75))
    fig_vuln_map.update_layout(
        height=680,
        margin={"r": 0, "t": 60, "l": 0, "b": 0},
        coloraxis_colorbar=dict(title="Vulnerabilidade"),
    )
    fig_vuln_map.show()
    print(f"Mapa renderizado: {len(_map_df):,} municípios plotados")


---

## 11. Exportação para Dashboard — Power BI / Tableau

### 11.1 Justificativa Metodológica — Comunicação de Resultados

Ferramentas de BI como **Power BI** e **Tableau** permitem dashboards interativos para stakeholders (Kimball & Ross, 2013). Preparamos CSVs agregados com:
- Resumo por estado (sanções, IDH, transferências).
- Resumo por cluster (perfil socioeconômico).
- Lista de municípios prioritários para auditoria (top 50 por vulnerabilidade).


In [ ]:
df_correlations.to_csv(output_dir / 'correlation_by_cluster.csv', index=False)

sample_cols = [c for c in ['codigo_municipio', 'nome_municipio', 'codigo_estado', 'nome_estado',
                           'cluster', 'category', 'rank_in_cluster', 'vulnerability_index',
                           'sancoes_por_milhao_brl_transferencias', 'renda_media_real_2022_brl_2022',
                           'taxa_alfabetizacao_2022', 'total_transferencias', 'num_sancoes']
               if c in df_sample.columns]
df_sample[sample_cols].to_csv(output_dir / 'representative_sample_cities.csv', index=False)
df_sample[sample_cols].to_csv(output_dir / 'amostra_representativa_cidades.csv', index=False)

numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
summary_cols = [c for c in ['vulnerability_index', CORRUPTION_VAR,
                            'renda_media_real_2022_brl_2022', 'taxa_alfabetizacao_2022']
                if c in numeric_cols]
cluster_summary = df_analysis.groupby('cluster')[summary_cols].agg(
    ['count', 'mean', 'std', 'min', 'max']).round(3)
cluster_summary.to_csv(output_dir / 'cluster_summary.csv')
cluster_summary.to_csv(output_dir / 'resumo_por_cluster.csv')

with open(output_dir / 'representative_sample.json', 'w', encoding='utf-8') as f:
    json.dump(df_sample.to_dict('records'), f, ensure_ascii=False, indent=2, default=str)

print("Dashboard exports:")
for fname in ['correlation_by_cluster.csv', 'representative_sample_cities.csv',
              'amostra_representativa_cidades.csv', 'cluster_summary.csv',
              'resumo_por_cluster.csv', 'representative_sample.json']:
    fp = output_dir / fname
    if fp.exists():
        print(f"  [OK] {fname:<42s} ({fp.stat().st_size:,} bytes)")

# ── Dashboard Interativo ──────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_dash = df_analysis[df_analysis["vulnerability_index"].notna()].copy()
_dash["cluster_label"] = _dash["cluster"].apply(
    lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Desconhecido"
)
_clusters = sorted(_dash["cluster"].dropna().unique())
_palette = px.colors.qualitative.Set2

fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Índice de Vulnerabilidade por Cluster",
        "Sanções/Milhão BRL vs Renda por Cluster",
        "Heatmap Pearson r (Cluster × Variável IDH)",
        "Top 20 Municípios Mais Vulneráveis",
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.12,
)

# 1. Box: vulnerabilidade por cluster
for i, c in enumerate(_clusters):
    fig_dash.add_trace(
        go.Box(
            y=_dash[_dash["cluster"] == c]["vulnerability_index"],
            name=f"Cluster {int(c)}",
            marker_color=_palette[i % len(_palette)],
            showlegend=False,
        ),
        row=1, col=1,
    )
fig_dash.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# 2. Scatter: sanções vs renda
for i, c in enumerate(_clusters):
    _s = _dash[_dash["cluster"] == c]
    _inc_col = "renda_media_real_2022_brl_2022"
    if _inc_col in _s.columns:
        fig_dash.add_trace(
            go.Scatter(
                x=_s[_inc_col],
                y=_s[CORRUPTION_VAR],
                mode="markers",
                marker=dict(size=4, opacity=0.5, color=_palette[i % len(_palette)]),
                name=f"Cluster {int(c)}",
                text=_s.get("nome_municipio", _s.index.astype(str)),
                hovertemplate="%{text}<br>Renda: %{x:,.0f}<br>Sanções/M: %{y:.2f}<extra></extra>",
            ),
            row=1, col=2,
        )

# 3. Heatmap de correlação
if len(df_correlations) > 0:
    _idh_label = {
        "renda_media_real_2022_brl_2022": "Renda",
        "taxa_alfabetizacao_2022": "Alfab.",
        "renda_media_2022": "Renda (bruta)",
    }
    _piv = df_correlations.pivot_table(index="cluster", columns="hdi_variable", values="pearson_r")
    _piv.columns = [_idh_label.get(c, c) for c in _piv.columns]
    fig_dash.add_trace(
        go.Heatmap(
            z=_piv.values,
            x=list(_piv.columns),
            y=[f"Cluster {int(c)}" for c in _piv.index],
            colorscale="RdBu", zmid=0, zmin=-0.5, zmax=0.5,
            text=_piv.values.round(3), texttemplate="%{text}",
            showscale=False,
        ),
        row=2, col=1,
    )

# 4. Top-20 municípios mais vulneráveis
_name_col = next((c for c in ["nome_municipio"] if c in _dash.columns), None)
_state_col = next((c for c in ["nome_estado"] if c in _dash.columns), None)
_top20 = _dash.nlargest(20, "vulnerability_index").copy()
_label = (
    (_top20[_name_col] + " (" + _top20[_state_col] + ")")
    if _name_col and _state_col
    else _top20.index.astype(str)
)
fig_dash.add_trace(
    go.Bar(
        x=_top20["vulnerability_index"],
        y=_label,
        orientation="h",
        marker_color="crimson",
        text=_top20["cluster_label"],
        textposition="outside",
        showlegend=False,
    ),
    row=2, col=2,
)

fig_dash.update_layout(
    height=900,
    title_text="Dashboard Consolidado — Análise de Vulnerabilidade por Cluster",
    title_font_size=16,
    showlegend=True,
    legend=dict(title="Cluster", x=1.01),
)
fig_dash.update_xaxes(title_text="Renda Média (BRL)", row=1, col=2)
fig_dash.update_yaxes(title_text="Sanções / Milhão BRL", row=1, col=2)
fig_dash.update_xaxes(title_text="Índice de Vulnerabilidade", row=2, col=2)
fig_dash.show()


---

## 12. Conclusões

### 12.1 Principais Achados

1. **Arquitetura Medallion implementada** — Pipeline de dados completo (Bronze, Silver, Gold) materializado em S3, documentado e reprodutível.
2. **Cobertura completa** — 5.570 municípios (99.9% do Brasil) e 27 estados.
3. **Correlação corrupção-IDH é heterogênea** — Não há correlação uniforme; efeitos variam por cluster e região.
4. **Clusters socioeconômicos identificados** — K=4 captura arquétipos: grandes centros urbanos, municípios em transição, áreas rurais de baixa renda, regiões de desenvolvimento médio.
5. **Municípios vulneráveis identificados** — Índice combinado destaca municípios com alta razão sanções/transferências E baixos indicadores de IDH.

### 12.2 Limitações

- **Dados de corte transversal** (2022) — Não podemos estabelecer causalidade estrita; associações são correlacionais.
- **Proxy de corrupção imperfeito** — Sanções detectadas dependem de capacidade institucional de auditagem (Ferraz & Finan, 2008; Olken & Pande, 2012).
- **Endogeneidade** — Regressões OLS são associacionais; variáveis omitidas (qualidade da gestão municipal, clima político local) podem confundir.
- **Viés de seleção em sanções** — Municípios com maior visibilidade (maior população, mais transferências) são mais auditados.

### 12.3 Implicações para Políticas Públicas

- **Priorização de auditorias** — O índice de vulnerabilidade pode informar alocação de recursos do TCU (TCU, 2023).
- **Intervenções diferenciadas** — Clusters distintos requerem estratégias diferentes; políticas uniformes são subótimas.
- **Transparência e dados abertos** — A disponibilidade de dados (LAI, 2011; LGPD, 2018) viabilizou esta pesquisa e deve ser expandida.

### 12.4 Trabalhos Futuros

- **Painel de dados longitudinais** — Coleta de séries temporais para aplicação de DiD (Diferenças-em-Diferenças) e análise de eventos (Angrist & Pischke, 2009).
- **Modelagem espacial** — Incorporar dependência espacial via modelos SAR/SEM (Anselin, 1988).
- **Enriquecimento com outras fontes** — Integrar dados do SICONFI, SIGMUN, e bases de gastos municipais detalhados.


---

## 13. Referências (Formato ABNT)

### 13.1 Fundamentos de Data Warehousing e Arquitetura de Dados

ARMBRUST, M. et al. **Lakehouse: A New Generation of Open Platforms**. Databricks, 2020.

DATABRICKS. **Medallion Architecture: Best Practices**. Databricks, 2023.

INMON, W. H. **Building the Data Warehouse**. 4. ed. Wiley, 2005.

KIMBALL, R.; ROSS, M. **The Data Warehouse Toolkit**. 3. ed. Wiley, 2013.

KLEPPMANN, M. **Designing Data-Intensive Applications**. O'Reilly, 2017.

### 13.2 Estatística, Econometria e Análise de Dados

ANGRIST, J. D.; PISCHKE, J.-S. **Mostly Harmless Econometrics**. Princeton UP, 2009.

ANSELIN, L. **Spatial Econometrics: Methods and Models**. Springer, 1988.

BREUSCH, T. S.; PAGAN, A. R. A Simple Test for Heteroscedasticity. **Econometrica**, v. 47, n. 5, p. 1287-1294, 1979.

COHEN, J.; COHEN, P.; WEST, S. G.; AIKEN, L. S. **Applied Multiple Regression**. 3. ed. Erlbaum, 2003.

FIELD, A. **Discovering Statistics Using SPSS**. 5. ed. Sage, 2017.

FOX, J. **Applied Regression Analysis and Generalized Linear Models**. 3. ed. Sage, 2015.

MONTGOMERY, D. C.; PECK, E. A.; VINING, G. G. **Introduction to Linear Regression Analysis**. 5. ed. Wiley, 2012.

TUKEY, J. W. **Exploratory Data Analysis**. Reading, MA: Addison-Wesley, 1977.

WOOLDRIDGE, J. M. **Econometric Analysis of Cross Section and Panel Data**. 2. ed. MIT Press, 2020.

### 13.3 Machine Learning e Análise Multivariada

BREIMAN, L. Random Forests. **Machine Learning**, v. 45, n. 1, p. 5-32, 2001.

GÉRON, A. **Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow**. 3. ed. O'Reilly, 2022.

HAIR, J. F. et al. **Multivariate Data Analysis**. 8. ed. Cengage, 2018.

HASTIE, T.; TIBSHIRANI, R.; FRIEDMAN, J. **The Elements of Statistical Learning**. 2. ed. Springer, 2009.

JAMES, G.; WITTEN, D.; HASTIE, T.; TIBSHIRANI, R. **An Introduction to Statistical Learning**. 2. ed. Springer, 2021.

JOLLIFFE, I. T. **Principal Component Analysis**. 2. ed. Springer, 2002.

KUHN, M.; JOHNSON, K. **Applied Predictive Modeling**. Springer, 2013.

MACQUEEN, J. B. Some Methods for Classification and Analysis. **Proceedings of the Fifth Berkeley Symposium**, v. 1, p. 281-297, 1967.

MOLNAR, C. **Interpretable Machine Learning**. 2. ed. 2022.

ZOU, H.; HASTIE, T. Regularization and Variable Selection via the Elastic Net. **Journal of the Royal Statistical Society, Series B**, v. 67, n. 2, p. 301-320, 2005.

### 13.4 Administração Pública e Corrupção

AFONSO, A.; SCHUKNECHT, L.; TANZI, V. Public Sector Efficiency. **Public Choice**, v. 123, n. 3-4, p. 321-347, 2005.

FERRAZ, C.; FINAN, F. Exposing Corrupt Politicians. **Quarterly Journal of Economics**, v. 123, n. 2, p. 703-745, 2008.

OLKEN, B. A.; PANDE, R. Corruption in Developing Countries. **Annual Review of Economics**, v. 4, p. 479-509, 2012.

ROSE-ACKERMAN, S.; PALIFKA, B. J. **Corruption and Government**. 2. ed. Cambridge UP, 2016.

### 13.5 Setor Público Brasileiro

BRASIL. **Lei nº 12.527/2011 — Lei de Acesso à Informação**. Brasília: Presidência da República, 2011.

BRASIL. **Lei nº 13.709/2018 — LGPD**. Brasília: Presidência da República, 2018.

CGU. **Portal da Transparência do Governo Federal**. Brasília: CGU, 2024.

IBGE. **Censo Demográfico 2010**. Rio de Janeiro: IBGE, 2010.

IBGE. **Censo Demográfico 2022**. Rio de Janeiro: IBGE, 2022.

IBGE. **API SIDRA**. Rio de Janeiro: IBGE, 2024.

STN. **SICONFI**. Brasília: STN, 2024.

TCU. **Referencial de Combate à Fraude e Corrupção**. Brasília: TCU, 2023.

PNUD. **Human Development Report 2020**. New York: UNDP, 2020.

### 13.6 Geoespacial e Reprodutibilidade

ANSELIN, L. **Spatial Econometrics: Methods and Models**. Springer, 1988.

BIVAND, R. S.; PEBESMA, E.; GÓMEZ-RUBIO, V. **Applied Spatial Data Analysis with R**. 2. ed. Springer, 2013.

GANDRUD, C. **Reproducible Research with R and RStudio**. 3. ed. CRC Press, 2020.

QGIS DEVELOPMENT TEAM. **QGIS Geographic Information System**. OSGeo, 2024.

WICKHAM, H. Tidy Data. **Journal of Statistical Software**, v. 59, n. 10, 2014.

WICKHAM, H.; GROLEMUND, G. **R for Data Science**. O'Reilly, 2017.

### 13.7 Stack Científico Python

HARRIS, C. R. et al. Array programming with NumPy. **Nature**, v. 585, p. 357-362, 2020.

HUNTER, J. D. Matplotlib: A 2D Graphics Environment. **Computing in Science & Engineering**, v. 9, n. 3, p. 90-95, 2007.

MCKINNEY, W. Data Structures for Statistical Computing in Python. **Proceedings of the 9th Python in Science Conference**, p. 56-61, 2010.

MCKINNEY, W. **Python for Data Analysis**. 3. ed. O'Reilly, 2022.

SEABOLD, S.; PERKTOLD, J. statsmodels: Econometric and Statistical Modeling. **Proceedings of the 9th Python in Science Conference**, p. 92-96, 2010.

VANDERPLAS, J. **Python Data Science Handbook**. O'Reilly, 2016.

WASKOM, M. L. seaborn: statistical data visualization. **Journal of Open Source Software**, v. 6, n. 60, 2021.

---

## Fim do Notebook

Este notebook constitui a **evidência empírica completa** do TCC. Todos os dados, código e conclusões são autocontidos e reprodutíveis.
